In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Create Data Samples") \
    .getOrCreate()

26/05/21 19:23:52 WARN Utils: Your hostname, bdlc-021 resolves to a loopback address: 127.0.1.1; using 10.176.129.84 instead (on interface ens3)
26/05/21 19:23:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 19:23:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/21 19:24:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
#Raw Parquet

raw_df = spark.read.parquet("/taxi/raw/2021/*.parquet")

raw_sample = raw_df.limit(1000)

raw_sample.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet("file:///home/cluster/BDLC/data_sample/yellow_taxi_raw_sample_1000_parquet")

In [2]:
# Balanced Clean Parquet Sample
# Pro Jahr-Monat-Kombination werden 1000 zufällige Zeilen genommen.
# Dadurch enthält das Sample alle Monate von 2015 bis 2021.

from pyspark.sql import functions as F
from functools import reduce

clean_df = spark.read.parquet("/taxi/clean")

ROWS_PER_MONTH = 1000
YEARS = range(2015, 2022)
MONTHS = range(1, 13)

sample_parts = []

for year in YEARS:
    for month in MONTHS:
        print(f"Sampling {year}-{month:02d}")
        
        part = clean_df \
            .filter((F.col("file_year") == year) & (F.col("file_month") == month)) \
            .orderBy(F.rand(seed=42)) \
            .limit(ROWS_PER_MONTH)
        
        sample_parts.append(part)

clean_sample_balanced = reduce(lambda df1, df2: df1.unionByName(df2), sample_parts)

print("Sample count:", clean_sample_balanced.count())

clean_sample_balanced.coalesce(1) \
    .write \
    .mode("overwrite") \
    .parquet("file:///home/cluster/BDLC/data_sample/yellow_taxi_clean_sample_84000_parquet")

Sampling 2015-01
Sampling 2015-02
Sampling 2015-03
Sampling 2015-04
Sampling 2015-05
Sampling 2015-06
Sampling 2015-07
Sampling 2015-08
Sampling 2015-09
Sampling 2015-10
Sampling 2015-11
Sampling 2015-12
Sampling 2016-01
Sampling 2016-02
Sampling 2016-03
Sampling 2016-04
Sampling 2016-05
Sampling 2016-06
Sampling 2016-07
Sampling 2016-08
Sampling 2016-09
Sampling 2016-10
Sampling 2016-11
Sampling 2016-12
Sampling 2017-01
Sampling 2017-02
Sampling 2017-03
Sampling 2017-04
Sampling 2017-05
Sampling 2017-06
Sampling 2017-07
Sampling 2017-08
Sampling 2017-09
Sampling 2017-10
Sampling 2017-11
Sampling 2017-12
Sampling 2018-01
Sampling 2018-02
Sampling 2018-03
Sampling 2018-04
Sampling 2018-05
Sampling 2018-06
Sampling 2018-07
Sampling 2018-08
Sampling 2018-09
Sampling 2018-10
Sampling 2018-11
Sampling 2018-12
Sampling 2019-01
Sampling 2019-02
Sampling 2019-03
Sampling 2019-04
Sampling 2019-05
Sampling 2019-06
Sampling 2019-07
Sampling 2019-08
Sampling 2019-09
Sampling 2019-10
Sampling 2019-

Sample count: 84000


In [3]:
ls -lh /home/cluster/BDLC/data_sample

total 7.3M
-rw-r--r-- 1 cluster cluster 4.2K May 19 19:23 01_create_sample_csv.ipynb
-rw-r--r-- 1 cluster cluster 7.1K May 21 19:25 02_create_sample_parquet.ipynb
-rw-r--r-- 1 cluster cluster 7.2M May 19 19:22 yellow_taxi_clean_sample_50000.csv
drwxr-xr-x 2 cluster cluster 4.0K May 19 19:31 yellow_taxi_clean_sample_50000_parquet/
drwxr-xr-x 2 cluster cluster 4.0K May 21 19:25 yellow_taxi_clean_sample_84000_parquet/
-rw-r--r-- 1 cluster cluster 103K May 19 19:21 yellow_taxi_raw_sample_1000.csv
drwxr-xr-x 2 cluster cluster 4.0K May 19 19:31 yellow_taxi_raw_sample_1000_parquet/


In [4]:
spark.stop()